In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('../data/raw/ericsson_patent_rich_dataset.csv')
df.head()

,patent_id,patent_type,patent_date,patent_title,year,quarter,month,day_of_week,day_of_year,quarter_end,...,patent_count_lag4,patent_count_roll4_mean,patent_count_roll8_mean,patent_count_qoq,patent_count_yoy,target_patent_count_current,target_patent_count_next_quarter,years_since_first_patent,patent_seq_global,year_patent_count
0,3934093,utility,1976-01-20,Transit exchange for asynchronous data with un...,1976,1,1,1,20,1976-03-31,...,324.0,347.75,337.375,0.023873,0.074324,10,5.0,0,1,36
1,3936612,utility,1976-02-03,Control and holding circuit for relay matrices,1976,1,2,1,34,1976-03-31,...,324.0,347.75,337.375,0.023873,0.074324,10,5.0,0,3,36
2,3936781,utility,1976-02-03,Selecting device for crossbar switches,1976,1,2,1,34,1976-03-31,...,324.0,347.75,337.375,0.023873,0.074324,10,5.0,0,2,36
3,3938025,utility,1976-02-10,DC to DC converter for producing two individua...,1976,1,2,1,41,1976-03-31,...,324.0,347.75,337.375,0.023873,0.074324,10,5.0,0,5,36
4,3938040,utility,1976-02-10,Apparatus for measuring a signal voltage inclu...,1976,1,2,1,41,1976-03-31,...,324.0,347.75,337.375,0.023873,0.074324,10,5.0,0,4,36


In [3]:
print(df.shape)
df.info()

(30118, 55)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30118 entries, 0 to 30117
Data columns (total 55 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   patent_id                         30118 non-null  object 
 1   patent_type                       30118 non-null  object 
 2   patent_date                       30118 non-null  object 
 3   patent_title                      30118 non-null  object 
 4   year                              30118 non-null  int64  
 5   quarter                           30118 non-null  int64  
 6   month                             30118 non-null  int64  
 7   day_of_week                       30118 non-null  int64  
 8   day_of_year                       30118 non-null  int64  
 9   quarter_end                       30118 non-null  object 
 10  title_len_chars                   30118 non-null  int64  
 11  title_len_words                   30118 non-null  int64

In [4]:
df.isna().sum()[df.isna().sum() > 0]

target_patent_count_next_quarter    408
dtype: int64

In [5]:
#Corregir tipos de datos
df['patent_date'] = pd.to_datetime(df['patent_date'])
df['quarter_end'] = pd.to_datetime(df['quarter_end'])

df['patent_type'] = df['patent_type'].astype('category')
df['tech_era'] = df['tech_era'].astype('category')

df.dtypes[['patent_date', 'quarter_end', 'patent_type', 'tech_era']]

patent_date    datetime64[ns]
quarter_end    datetime64[ns]
patent_type          category
tech_era             category
dtype: object

In [6]:
#Corregir duplicados y consistencia
print("Duplicados exactos:", df.duplicated().sum())
print("patent_id duplicados:", df['patent_id'].duplicated().sum())

Duplicados exactos: 0
patent_id duplicados: 0


In [7]:
mismatch_year = (df['patent_date'].dt.year != df['year']).sum()
mismatch_quarter = (df['patent_date'].dt.quarter != df['quarter']).sum()
print("Inconsistencias en year:", mismatch_year)
print("Inconsistencias en quarter:", mismatch_quarter)

Inconsistencias en year: 0
Inconsistencias en quarter: 0


In [8]:
n_trimestres = df[['year','quarter']].drop_duplicates().shape[0]
print("Trimestres únicos:", n_trimestres)

Trimestres únicos: 199


In [9]:
#Investigar los nulos del target
nulos = df[df['target_patent_count_next_quarter'].isna()]
print("Filas con target nulo:", len(nulos))
nulos[['year', 'quarter']].drop_duplicates()


Filas con target nulo: 408


,year,quarter
29710,2025,3


In [10]:
#Detectar qué columnas son "por trimestre" vs "por patente"
cols_trimestre = []
cols_patente = []

for col in df.columns:
    if col in ['year', 'quarter']:
        continue
    n_valores_unicos = df.groupby(['year', 'quarter'])[col].nunique()
    if (n_valores_unicos <= 1).all():
        cols_trimestre.append(col)
    else:
        cols_patente.append(col)

print("Columnas a nivel TRIMESTRE:", len(cols_trimestre))
print(cols_trimestre)
print()
print("Columnas a nivel PATENTE:", len(cols_patente))
print(cols_patente)

Columnas a nivel TRIMESTRE: 29
['day_of_week', 'quarter_end', 'tech_era', 'patent_count', 'patent_unique', 'utility_share', 'design_share', 'avg_title_words', 'avg_keyword_score', 'kw_5g_count', 'kw_ai_ml_count', 'kw_cloud_edge_count', 'kw_security_count', 'kw_iot_count', 'kw_network_count', 'kw_energy_count', 'kw_antenna_count', 'kw_data_count', 'patent_count_lag1', 'patent_count_lag2', 'patent_count_lag4', 'patent_count_roll4_mean', 'patent_count_roll8_mean', 'patent_count_qoq', 'patent_count_yoy', 'target_patent_count_current', 'target_patent_count_next_quarter', 'years_since_first_patent', 'year_patent_count']

Columnas a nivel PATENTE: 24
['patent_id', 'patent_type', 'patent_date', 'patent_title', 'month', 'day_of_year', 'title_len_chars', 'title_len_words', 'title_has_number', 'title_has_acronym', 'kw_5g', 'kw_ai_ml', 'kw_cloud_edge', 'kw_security', 'kw_iot', 'kw_network', 'kw_energy', 'kw_antenna', 'kw_data', 'keyword_score', 'is_utility', 'is_design', 'is_other_type', 'patent_s

In [12]:
#Crear los dos dataframes finales
patent_level = df[cols_patente + ['year', 'quarter']].copy()
patent_level = patent_level.sort_values('patent_date').reset_index(drop=True)

quarterly_level = df[cols_trimestre + ['year', 'quarter']].copy()
quarterly_level = quarterly_level.drop_duplicates(subset=['year', 'quarter'])
quarterly_level = quarterly_level.sort_values(['year', 'quarter']).reset_index(drop=True)

print("patent_level:", patent_level.shape)
print("quarterly_level:", quarterly_level.shape)

patent_level: (30118, 26)
quarterly_level: (199, 31)


In [13]:
#Exportacion 
patent_level.to_csv('../data/processed/patent_level_clean.csv', index=False)
quarterly_level.to_csv('../data/processed/quarterly_level_clean.csv', index=False)

print("Archivos guardados correctamente")

Archivos guardados correctamente
